In [417]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
import operator

In [418]:
load_dotenv()

True

In [419]:
class EvaluatorSchema(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation of the tweet")
    feedback: str = Field(..., description="The Feedback for the tweet")

In [420]:
generator_llm = ChatGroq(model="openai/gpt-oss-20b")
evaluator_llm = ChatGroq(model="openai/gpt-oss-20b")
structured_evaluator_llm = evaluator_llm.with_structured_output(EvaluatorSchema)
optimizer_llm = ChatGroq(model="openai/gpt-oss-20b")

In [421]:
class XPostState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int
    tweet_history: Annotated[list, operator.add]
    feedback_history: Annotated[list, operator.add]

In [422]:
def generate_content(state: XPostState):
    prompt = [
        SystemMessage(content='You are a funny and clever Twitter/X influencer.'),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    response =  generator_llm.invoke(prompt).content
    return {"tweet": response, "tweet_history": [response]}


    

In [423]:
def evaluate(state: XPostState):
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format. Evaluate the tweet and respond in JSON format."),
        HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
    ]
    
    response =  structured_evaluator_llm.invoke(messages)

    return {"evaluation": response.evaluation, "feedback": response.feedback, "feedback_history": [response.feedback]}

In [424]:
def optimize(state: XPostState):
    messages = [
        SystemMessage(
            content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content

    iteration = state["iteration"] + 1

    return {"tweet": response, "iteration": iteration, "tweet_history": [response], "feedback": state['feedback']}

In [425]:
def route_evaluations(state: XPostState) -> Literal["approved", "needs_improvement"]:
    if state["evaluation"] == "approved" or state['iteration'] >= state["max_iteration"]:
        return "approved"
    else:
        return "needs_improvement"

In [426]:
graph = StateGraph(XPostState)

graph.add_node("generate_content", generate_content)
graph.add_node("evaluate", evaluate)
graph.add_node("optimize", optimize)

graph.add_edge(START, "generate_content")
graph.add_edge("generate_content", "evaluate")
graph.add_conditional_edges("evaluate", route_evaluations, {"approved": END, "needs_improvement": "optimize"})
graph.add_edge("optimize", "evaluate")

workflow = graph.compile()

In [427]:
initial_state = {
    "topic": "Ai replacing software engineer",
    "iteration": 1,
    "max_iteration": 3
}

result = workflow.invoke(initial_state)

In [428]:
# for tweet in result["tweet_history"]:
#     print(tweet)
result

{'topic': 'Ai replacing software engineer',
 'tweet': 'Just watched an AI write a full‑stack app in 3 hours. Meanwhile, I’m still trying to remember if I need to commit or push. Next thing you know, the AI will file my taxes, water my plants, and still ask for a coffee break. #FutureOfWork',
 'evaluation': 'approved',
 'feedback': 'This tweet hits the sweet spot of tech‑humor: the juxtaposition of a hyper‑efficient AI versus the writer’s own Git hiccups feels fresh and relatable. The pacing is tight—no filler, just a quick narrative arc that ends on a punchy hashtag, making it scroll‑stopper material. The humor is light, self‑deprecating, and likely to resonate with developers and AI enthusiasts alike, giving it good virality potential. Minor tweak: tightening the wording a bit could shave a few characters, but overall it’s a solid, shareable tweet.',
 'iteration': 1,
 'max_iteration': 3,
 'tweet_history': ['Just watched an AI write a full‑stack app in 3 hours. Meanwhile, I’m still try